In [ ]:
!pip install xgboost

In [ ]:
!pip install scikit-learn

In [1]:
import polars as pl
import xgboost as xgb
from sklearn.metrics import classification_report, average_precision_score, confusion_matrix
import os
import gc

# 1. 고도화된 피처 파일 로드
file_path = "/home/tracerofjageum/aml_features_final_advanced.parquet"
print("📂 데이터 로딩 중...")
df = pl.read_parquet(file_path)

# 2. 시간 순서대로 6:2:2 분할 (Data Leakage 방지)
print("✂️ 시간 기반 데이터 분할 중 (Train: 60%, Val: 20%, Test: 20%)...")
df = df.sort("time_group")
total_len = len(df)
train_end = int(total_len * 0.6)
val_end = int(total_len * 0.8)

train_df = df.slice(0, train_end)
val_df = df.slice(train_end, val_end - train_end)
test_df = df.slice(val_end, total_len - val_end)

# 학습에 사용할 피처 선택 (ID와 타겟 제외)
features = [col for col in df.columns if col not in ["account_id", "time_group", "is_laundering"]]
target = "is_laundering"

print(f"📊 학습에 사용하는 피처 ({len(features)}개): {features}")

# 3. XGBoost용 데이터 형식으로 변환
print("🚀 DMatrix 변환 중...")
dtrain = xgb.DMatrix(train_df.select(features).to_pandas(), label=train_df[target].to_pandas())
dval = xgb.DMatrix(val_df.select(features).to_pandas(), label=val_df[target].to_pandas())
dtest = xgb.DMatrix(test_df.select(features).to_pandas(), label=test_df[target].to_pandas())

# 메모리 정리
del df, train_df, val_df; gc.collect()

# 4. 모델 파라미터 설정 (타겟 불균형 대응)
# scale_pos_weight: (정상 거래 수 / 세탁 거래 수) -> 약 424
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',      # 불균형 데이터에는 AUC-PR이 가장 정확함
    'max_depth': 6,
    'learning_rate': 0.1,
    'scale_pos_weight': 424,    # 범죄자를 놓치지 않기 위한 가중치
    'tree_method': 'hist',      # 대용량 데이터 처리용
    'device': 'cpu'             # GPU가 있다면 'cuda'로 변경 가능
}

# 5. 모델 학습 (Early Stopping 적용)
print("🏋️ 모델 학습 시작...")
model = xgb.train(
    params,
    dtrain,
    num_boost_round=500,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=10,
    verbose_eval=10
)

# 6. 테스트 데이터 평가
print("\n🎯 테스트 데이터 평가 결과:")
preds = model.predict(dtest)
y_pred = [1 if p > 0.5 else 0 for p in preds]
y_true = dtest.get_label()

print(classification_report(y_true, y_pred))
print(f"Average Precision (AUPRC): {average_precision_score(y_true, preds):.4f}")

# 7. 피처 중요도 확인 (어떤 피처가 범인을 잘 잡았나?)
print("\n🔥 피처 중요도 (Top 10):")
importance = model.get_score(importance_type='gain')
sorted_importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)
for f, s in sorted_importance[:10]:
    print(f"{f}: {s:.2f}")

📂 데이터 로딩 중...
✂️ 시간 기반 데이터 분할 중 (Train: 60%, Val: 20%, Test: 20%)...
📊 학습에 사용하는 피처 (27개): ['cnt_1h', 'time_delta_mean', 'time_delta_std', 'min_inter_tx_gap', 'cnt_night', 'cnt_weekend', 'sum_1h', 'max_1h', 'std_1h', 'sum_out_1h', 'sum_in_1h', 'cnt_small_tx', 'cnt_risk_country', 'ratio_cross_border', 'degree_1h', 'entity_acct_cnt', 'cnt_risk_format', 'benford_1_ratio', 'avg_log_amount', 'sum_24h', 'cnt_24h', 'avg_7d', 'burst_ratio', 'in_out_balance_ratio', 'inter_arrival_cv', 'benford_deviation', 'ratio_risk_country']
🚀 DMatrix 변환 중...
🏋️ 모델 학습 시작...
[0]	train-aucpr:0.14969	val-aucpr:0.25741
[10]	train-aucpr:0.17114	val-aucpr:0.23651

🎯 테스트 데이터 평가 결과:
              precision    recall  f1-score   support

         0.0       1.00      0.94      0.97   5730724
         1.0       0.05      0.93      0.10     20537

    accuracy                           0.94   5751261
   macro avg       0.53      0.94      0.53   5751261
weighted avg       1.00      0.94      0.97   5751261

Average Precis

되게 공격적인 모델이네요 recall이 0.93.. 

In [2]:
# 8. V1 모델 Top-K 탐지 성능 분석 (전체 및 일자별)
import pandas as pd

print("\n🔍 V1 모델 Top-K 분석 시작...")

# 테스트 데이터의 예측 확률과 실제 라벨, 날짜 정보 결합
# test_df['time_group']에서 날짜 정보를 추출합니다.
v1_results = pd.DataFrame({
    'date': test_df['time_group'].dt.date().to_pandas(),
    'probability': preds,
    'actual': dtest.get_label()
})

# --- [8-1] 전체 Top-K 탐지 성능 ---
print("\n" + "="*50)
print(f"{'Top-K':<10} | {'실제 범인 수':<12} | {'정밀도(Precision@K)':<20}")
print("-" * 50)

# 확률 기준 내림차순 정렬
v1_sorted = v1_results.sort_values(by='probability', ascending=False).reset_index(drop=True)

ks = [100, 500, 1000, 5000, 10000]
for k in ks:
    top_k = v1_sorted.head(k)
    hits = int(top_k['actual'].sum())
    precision = hits / k
    print(f"Top {k:<5} | {hits:<12} | {precision:.2%}")

# --- [8-2] 일자별 Top-100 탐지 성능 ---
print("\n" + "="*50)
print("📊 [V1 모델: 일자별 Top-100 탐지 리포트]")
print("-" * 50)

daily_report = []
# 날짜별로 그룹화하여 상위 100명 추출
for date, group in v1_results.groupby('date'):
    # 해당 날짜 전체 범인 수
    day_total_criminals = int(group['actual'].sum())
    
    # 해당 날짜 내에서 확률순 정렬
    top_100 = group.sort_values(by='probability', ascending=False).head(100)
    hits = int(top_100['actual'].sum())
    precision = hits / 100
    
    daily_report.append({
        'Date': date,
        'Daily_Total': day_total_criminals,
        'Hits@100': hits,
        'Prec@100': f"{precision:.1%}"
    })

# 리포트 출력
daily_df = pd.DataFrame(daily_report)
print(daily_df.to_string(index=False))
print("="*50)

# 평균 성능 요약
avg_prec_100 = daily_df['Hits@100'].mean()
print(f"\n💡 V1 모델 일평균 Top-100 탐지 수: {avg_prec_100:.1f}명")


🔍 V1 모델 Top-K 분석 시작...

Top-K      | 실제 범인 수      | 정밀도(Precision@K)    
--------------------------------------------------
Top 100   | 48           | 48.00%
Top 500   | 201          | 40.20%
Top 1000  | 422          | 42.20%
Top 5000  | 3911         | 78.22%
Top 10000 | 5423         | 54.23%

📊 [V1 모델: 일자별 Top-100 탐지 리포트]
--------------------------------------------------
      Date  Daily_Total  Hits@100 Prec@100
2022-09-14         3608        32    32.0%
2022-09-15         4326        36    36.0%
2022-09-16         4549        42    42.0%
2022-09-17         2362       100   100.0%
2022-09-18         1808       100   100.0%
2022-09-19         1368       100   100.0%
2022-09-20          967       100   100.0%
2022-09-21          663       100   100.0%
2022-09-22          366       100   100.0%
2022-09-23          167       100   100.0%
2022-09-24          129       100   100.0%
2022-09-25          118       100   100.0%
2022-09-26           72        72    72.0%
2022-09-27           

### 튜닝을해봅시다

In [3]:
import xgboost as xgb
from sklearn.metrics import classification_report, average_precision_score

# 1. 파라미터 튜닝 (오탐 방어 모드)
params_tuned = {
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',
    'max_depth': 5,            # 트리 깊이를 줄여 일반화 능력 향상
    'learning_rate': 0.05,      # 학습률을 낮춰 더 세밀하게 학습
    'scale_pos_weight': 100,    # 가중치를 424 -> 100으로 하향 (오탐 감소 핵심)
    'min_child_weight': 10,     # 관측치가 적은 노드는 생성하지 않음 (노이즈 방지)
    'subsample': 0.8,           # 데이터 샘플링으로 다양성 확보
    'tree_method': 'hist',
    'device': 'cpu'
}

# 2. 모델 재학습
print("🏋️ 튜닝된 파라미터로 모델 재학습 시작...")
model_tuned = xgb.train(
    params_tuned,
    dtrain,
    num_boost_round=1000,       # 학습률을 낮췄으므로 라운드 수를 늘림
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=20,
    verbose_eval=50
)

# 3. 임계값(Threshold) 상향 조정 및 평가
print("\n🎯 튜닝 결과 평가 (Threshold 0.7 적용):")
preds_raw = model_tuned.predict(dtest)

# 확실한 경우(0.7 초과)만 1로 분류
threshold = 0.7
y_pred_tuned = [1 if p > threshold else 0 for p in preds_raw]
y_true = dtest.get_label()

print(classification_report(y_true, y_pred_tuned))
print(f"Average Precision (AUPRC): {average_precision_score(y_true, preds_raw):.4f}")

🏋️ 튜닝된 파라미터로 모델 재학습 시작...
[0]	train-aucpr:0.08047	val-aucpr:0.11824
[50]	train-aucpr:0.24229	val-aucpr:0.33043
[100]	train-aucpr:0.28509	val-aucpr:0.38294
[150]	train-aucpr:0.29811	val-aucpr:0.39405
[200]	train-aucpr:0.31243	val-aucpr:0.40872
[250]	train-aucpr:0.32054	val-aucpr:0.41919
[300]	train-aucpr:0.32569	val-aucpr:0.42402
[350]	train-aucpr:0.33042	val-aucpr:0.42891
[400]	train-aucpr:0.33396	val-aucpr:0.43231
[450]	train-aucpr:0.33680	val-aucpr:0.43475
[500]	train-aucpr:0.33966	val-aucpr:0.43681
[550]	train-aucpr:0.34127	val-aucpr:0.43793
[600]	train-aucpr:0.34282	val-aucpr:0.43899
[626]	train-aucpr:0.34312	val-aucpr:0.43881

🎯 튜닝 결과 평가 (Threshold 0.7 적용):
              precision    recall  f1-score   support

         0.0       1.00      0.99      0.99   5730724
         1.0       0.18      0.79      0.29     20537

    accuracy                           0.99   5751261
   macro avg       0.59      0.89      0.64   5751261
weighted avg       1.00      0.99      0.99   5751261

Av

In [4]:
import pandas as pd
import numpy as np

# 1. 예측값과 실제값 결합
test_results = pd.DataFrame({
    'probability': preds_raw,  # 모델의 예측 확률
    'actual': dtest.get_label() # 실제 라벨
})

# 2. 확률 기준 내림차순 정렬 (가장 수상한 놈이 위로)
test_results = test_results.sort_values(by='probability', ascending=False).reset_index(drop=True)

# 3. Top-K 설정 (조사 가능 인원 시나리오)
ks = [100, 500, 1000, 5000, 10000]
print(f"--- [Top-K 탐지 성능 평가] ---")
print(f"{'Top-K':<10} | {'실제 범인 수':<12} | {'정밀도(Precision@K)':<20}")
print("-" * 50)

for k in ks:
    top_k = test_results.head(k)
    # 수정된 부분: top_df -> top_k
    hits = top_k['actual'].sum() # Top-K 중 실제 범인(1)의 합
    precision_at_k = hits / k
    print(f"Top {k:<5} | {int(hits):<12} | {precision_at_k:.2%}")

print("-" * 50)

--- [Top-K 탐지 성능 평가] ---
Top-K      | 실제 범인 수      | 정밀도(Precision@K)    
--------------------------------------------------
Top 100   | 100          | 100.00%
Top 500   | 497          | 99.40%
Top 1000  | 987          | 98.70%
Top 5000  | 4523         | 90.46%
Top 10000 | 7239         | 72.39%
--------------------------------------------------


In [6]:
import pandas as pd
import numpy as np
import xgboost as xgb

# 1. V1 피처 리스트 정의 (first_model (1).ipynb에서 사용된 27개 피처)
v1_features = [
    'cnt_1h', 'time_delta_mean', 'time_delta_std', 'min_inter_tx_gap', 'cnt_night', 
    'cnt_weekend', 'sum_1h', 'max_1h', 'std_1h', 'sum_out_1h', 'sum_in_1h', 
    'cnt_small_tx', 'cnt_risk_country', 'ratio_cross_border', 'degree_1h', 
    'entity_acct_cnt', 'cnt_risk_format', 'benford_1_ratio', 'avg_log_amount', 
    'sum_24h', 'cnt_24h', 'avg_7d', 'burst_ratio', 'in_out_balance_ratio', 
    'inter_arrival_cv', 'benford_deviation', 'ratio_risk_country'
]

# 2. V1 모델 변수 확인 및 예측 (ipynb 파일의 변수명 'model' 또는 'model_tuned' 사용)
print("🔍 V1 모델 예측 및 일자별 분석 시작...")

# 메모리에 있는 모델 변수 찾기
if 'model' in globals():
    target_v1_model = model
    print("✅ 'model' 변수를 V1 모델로 인식했습니다.")
elif 'model_tuned' in globals():
    target_v1_model = model_tuned
    print("✅ 'model_tuned' 변수를 V1 모델로 인식했습니다.")
elif 'model_v1' in globals():
    target_v1_model = model_v1
else:
    raise NameError("V1 모델 변수(model)를 찾을 수 없습니다. V1 학습 셀을 먼저 실행해주세요.")

# 3. V1 전용 DMatrix 생성 (현재 데이터에서 V1 피처 27개만 선택)
# 이렇게 해야 V1 모델이 피처 개수 오류 없이 작동합니다.
dtest_v1 = xgb.DMatrix(test_df.select(v1_features).to_pandas(), label=test_df['is_laundering'].to_pandas())

# 4. 예측 수행
preds_v1 = target_v1_model.predict(dtest_v1)

# 5. 분석용 데이터프레임 생성
v1_daily_analysis = pd.DataFrame({
    'date': pd.to_datetime(test_df['time_group'].to_pandas()).dt.date,
    'prob': preds_v1,
    'label': test_df['is_laundering'].to_pandas()
})

# 6. 일자별 Top-K 평가 함수 및 리포트 출력
def get_v1_daily_report(df, top_ks=[50, 100, 200, 500]):
    results = []
    for date, day_data in df.groupby('date'):
        day_total = int(day_data['label'].sum())
        day_data = day_data.sort_values(by='prob', ascending=False)
        row = {'Date': date, 'Total_Criminals': day_total}
        for k in top_ks:
            top_k_data = day_data.head(k)
            hits = int(top_k_data['label'].sum())
            row[f'Prec@{k}'] = f"{(hits/k)*100:.1f}%"
        results.append(row)
    return pd.DataFrame(results)

v1_report_df = get_v1_daily_report(v1_daily_analysis)

print("\n" + "="*80)
print("📊 [V1 Baseline 모델: 일자별 탐지 정밀도 리포트]")
print(v1_report_df.to_string(index=False))
print("="*80)

# 7. 평균 성능 요약
print("\n💡 [V1 평균 성능 요약]")
for k in [50, 100, 200, 500]:
    avg_p = v1_report_df[f'Prec@{k}'].str.replace('%', '').astype(float).mean()
    print(f"평균 Precision@{k:3} : {avg_p:.2f}%")
    

🔍 V1 모델 예측 및 일자별 분석 시작...
✅ 'model' 변수를 V1 모델로 인식했습니다.

📊 [V1 Baseline 모델: 일자별 탐지 정밀도 리포트]
      Date  Total_Criminals Prec@50 Prec@100 Prec@200 Prec@500
2022-09-14             3608   34.0%    32.0%    33.5%    59.0%
2022-09-15             4326   36.0%    36.0%    36.5%    51.6%
2022-09-16             4549   42.0%    42.0%    49.5%    54.6%
2022-09-17             2362  100.0%   100.0%   100.0%   100.0%
2022-09-18             1808  100.0%   100.0%   100.0%   100.0%
2022-09-19             1368  100.0%   100.0%   100.0%   100.0%
2022-09-20              967  100.0%   100.0%   100.0%   100.0%
2022-09-21              663  100.0%   100.0%   100.0%   100.0%
2022-09-22              366  100.0%   100.0%   100.0%    73.2%
2022-09-23              167  100.0%   100.0%    83.5%    33.4%
2022-09-24              129  100.0%   100.0%    64.5%    25.8%
2022-09-25              118  100.0%   100.0%    59.0%    23.6%
2022-09-26               72  100.0%    72.0%    36.0%    14.4%
2022-09-27               28

In [8]:
print("\n🔥 피처 중요도 (Top 10):")
importance = model.get_score(importance_type='gain')
sorted_importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)
for f, s in sorted_importance[:]:
    print(f"{f}: {s:.2f}")


🔥 피처 중요도 (Top 10):
cnt_risk_format: 993179.50
ratio_cross_border: 706445.06
sum_1h: 362795.94
degree_1h: 242803.45
entity_acct_cnt: 242528.28
burst_ratio: 86531.74
sum_out_1h: 40213.93
cnt_1h: 30172.44
max_1h: 28659.73
sum_in_1h: 28070.84
cnt_night: 26737.06
time_delta_mean: 21183.34
cnt_weekend: 11081.26
min_inter_tx_gap: 10415.73
avg_log_amount: 4214.06
in_out_balance_ratio: 3424.02
std_1h: 2843.67
sum_24h: 1833.93
benford_deviation: 1281.01
time_delta_std: 1230.56
cnt_small_tx: 968.35
benford_1_ratio: 894.89
inter_arrival_cv: 304.95
